# BudgiBrain
## Built with LangGraph, Semantic Memory, and Episodic Memory

### Dependencies and Requirements

In [ ]:
# Insalling pip dependecies
%pip install -r requirements.txt

In [3]:
# Importing all packages
from langgraph.graph import StateGraph, END
from langchain.docstore.document import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_core.output_parsers import PydanticOutputParser
from langchain_groq import ChatGroq
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field
from typing import Optional, List, Dict, Any
from typing_extensions import TypedDict
from datetime import datetime
from IPython.display import display, clear_output
from dotenv import load_dotenv
from dataclasses import dataclass
import ipywidgets as widgets
from uuid import uuid4
import os
import re
import json


# Budget guidelines FAISS (self-contained)
BUDGET_FAISS_PATH = "../data/faiss_store_budget"

# Loading env variables
load_dotenv()

# Splitting of sample data into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=10)
# Setting up embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create/Load guidelines store
if os.path.exists(f"{BUDGET_FAISS_PATH}/index.faiss"):
    db_guidelines = FAISS.load_local(
        f"{BUDGET_FAISS_PATH}",
        embeddings=embedding_model,
        allow_dangerous_deserialization=True
    )
    print("✅ Loaded existing budget guidelines FAISS")
else:
    print("🔴 Unable to fetch budget guidelines FAISS")

# Initialize LLM
LLM = ChatGroq(
    model_name=os.environ.get("LITELLM_MODEL"),
    groq_api_key=os.environ.get("GROQ_API_KEY")
)


✅ Loaded existing budget guidelines FAISS


In [ ]:
# Collect profile/goals and propose budgets

@dataclass
class BudgetProfile:
    net_income: float # net income
    cost_of_living: str # high, medium, low
    household_size: int # no. of people in household

@dataclass
class FinancialGoal:
    name: str # name of financial goal
    target_amount: float # amount to be saved 
    months: int # no. of months reuired to save amount
    priority: int  # 1 - 5, 1 is highest

#TO DO : This would need to be fetched from the transaction memory
CATEGORIES = [
    "Rent", "Utilities", "Groceries", "Transport", "Health", "Subscriptions", "Shopping & Entertainment"
]

# Simple rules to convert guideline text into midpoints
GUIDE_DEFAULTS = {
    "high": {
        "Rent": (30, 40), "Utilities": (5, 8), "Groceries": (8, 12), "Transport": (5, 10),
        "Health": (3, 5), "Subscriptions": (2, 5), "Shopping & Entertainment": (5, 10)
    },
    "medium": {
        "Rent": (25, 35), "Utilities": (6, 10), "Groceries": (10, 15), "Transport": (8, 12),
        "Health": (4, 6), "Subscriptions": (2, 4), "Shopping & Entertainment": (4, 8)
    },
    "low": {
        "Rent": (20, 30), "Utilities": (5, 8), "Groceries": (10, 14), "Transport": (3, 7),
        "Health": (2, 4), "Subscriptions": (2, 4), "Shopping & Entertainment": (6, 12)
    }
}

ELASTICITY_ORDER = ["Shopping & Entertainment", "Subscriptions", "Transport", "Groceries", "Health"]
FLOORS = {"Groceries": 8, "Health": 3}


def midpoint(a: int, b: int) -> float:
    return (a + b) / 2.0


def propose_base_percentages(profile: BudgetProfile) -> Dict[str, float]:
    guide = GUIDE_DEFAULTS.get(profile.cost_of_living, GUIDE_DEFAULTS["medium"])
    base = {cat: midpoint(*guide[cat]) for cat in CATEGORIES}
    return base


def monthly_goal_savings(goals: List[FinancialGoal]) -> float:
    return sum(g.target_amount / max(g.months, 1) for g in goals)


def allocate_budget(profile: BudgetProfile, goals: List[FinancialGoal]) -> Dict[str, Any]:
    base_pct = propose_base_percentages(profile)
    total_base = sum(base_pct.values())

    # Start from base, clamp to <= 100%
    scale = min(100.0 / max(total_base, 1e-6), 1.0)
    for k in base_pct:
        base_pct[k] *= scale

    # Reserve savings for goals by reducing variable categories per elasticity
    income = profile.net_income
    required_savings = monthly_goal_savings(goals)

    # Convert to amounts
    allocations = {k: income * (p / 100.0) for k, p in base_pct.items()}

    # If needed, carve out savings
    if required_savings > 0:
        carved = 0.0
        for cat in ELASTICITY_ORDER:
            if carved >= required_savings:
                break
            floor_pct = FLOORS.get(cat, 0.0)
            floor_amt = income * (floor_pct / 100.0)
            available = max(allocations[cat] - floor_amt, 0.0)
            take = min(required_savings - carved, available)
            allocations[cat] -= take
            carved += take

        savings_shortfall = max(required_savings - carved, 0.0)
    else:
        savings_shortfall = 0.0

    plan = {
        "allocations": allocations,
        "required_savings": required_savings,
        "savings_shortfall": savings_shortfall,
        "income": income
    }
    return plan


def render_plan(plan: Dict[str, Any]) -> str:
    lines = []
    lines.append(f"Net income: {plan['income']:.2f}")
    lines.append("Proposed monthly allocations:")
    for cat in CATEGORIES:
        amt = plan["allocations"][cat]
        pct = (amt / max(plan['income'], 1e-6)) * 100
        lines.append(f"- {cat}: {amt:.2f} ({pct:.1f}%)")
    if plan["required_savings"] > 0:
        lines.append(f"Savings toward goals: {plan['required_savings']:.2f}")
        if plan["savings_shortfall"] > 0:
            lines.append(f"Shortfall: {plan['savings_shortfall']:.2f} — consider extending deadlines or tightening discretionary spend.")
    return "\n".join(lines)


In [ ]:
# Tool-enabled budget agent using LangGraph ToolNode
MEMORY_PATH = os.path.join(os.getcwd(), "data/budget_memory.json")

def _load_memory():
    try:
        with open(MEMORY_PATH, "r") as f:
            data = json.load(f)
        # reconstruct objects
        profile = data.get("profile")
        if profile:
            profile = BudgetProfile(**profile)
        goals = [FinancialGoal(**g) for g in data.get("goals", [])]
        plan = data.get("plan")
        return {"profile": profile, "goals": goals, "plan": plan}
    except Exception:
        return {"profile": None, "goals": [], "plan": None}


def _save_memory(ctx):
    try:
        serializable = {
            "profile": ctx["profile"].__dict__ if ctx.get("profile") else None,
            "goals": [g.__dict__ for g in ctx.get("goals", [])],
            "plan": ctx.get("plan"),
            "updated_at": datetime.utcnow().isoformat()
        }
        with open(MEMORY_PATH, "w") as f:
            json.dump(serializable, f, indent=2)
    except Exception as e:
        print(f"Warning: failed saving memory: {e}")


budget_context = _load_memory()

# Tools that operate on budget_context

def _parse_float(value) -> float:
    if isinstance(value, (int, float)):
        return float(value)
    s = str(value).strip().lower()
    # remove currency symbols and commas
    s = s.replace(",", " ")
    nums = re.findall(r"-?\d+(?:\.\d+)?", s)
    if not nums:
        raise ValueError(f"Could not parse float from: {value}")
    return float(nums[0])


def _parse_int(value) -> int:
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        return int(value)
    nums = re.findall(r"-?\d+", str(value))
    if not nums:
        raise ValueError(f"Could not parse int from: {value}")
    return int(nums[0])


def _parse_months(value) -> int:
    if isinstance(value, (int, float)):
        return int(value)
    s = str(value).strip().lower()
    # handle year-based durations
    m = re.search(r"(\d+(?:\.\d+)?)\s*(years?|yrs?|y)\b", s)
    if m:
        return int(round(float(m.group(1)) * 12))
    # handle month-based durations
    m = re.search(r"(\d+)\s*(months?|mos?|m)\b", s)
    if m:
        return int(m.group(1))
    # fallback: first integer found
    return _parse_int(value)


@tool
def set_profile_tool(net_income: str, cost_of_living: str, household_size: str) -> str:
    """Set the user's profile: net monthly income, cost_of_living in {low, medium, high}, and household size."""
    profile = BudgetProfile(
        net_income=_parse_float(net_income),
        cost_of_living=str(cost_of_living).lower(),
        household_size=_parse_int(household_size)
    )
    budget_context["profile"] = profile
    _save_memory(budget_context)
    return "Profile saved. You can now add goals or ask to 'propose budget'. No further action needed."
@tool
def add_goal_tool(name: str, target_amount: str, months: str, priority: str = "3") -> str:
    """Add a financial goal with a name, target_amount, months to reach, and priority (1 highest)."""
    goal = FinancialGoal(
        name=name,
        target_amount=_parse_float(target_amount),
        months=_parse_months(months),
        priority=_parse_int(priority)
    )
    budget_context.setdefault("goals", []).append(goal)
    _save_memory(budget_context)
    return f"Added goal '{name}'."
@tool
def propose_budget_tool() -> str:
    """Compute a proposed monthly budget based on the saved profile and goals."""
    profile = budget_context.get("profile")
    goals = budget_context.get("goals", [])
    if profile is None:
        return "Please set your profile first."
    plan = allocate_budget(profile, goals)
    budget_context["plan"] = plan
    _save_memory(budget_context)
    return render_plan(plan)

@tool
def adjust_category_tool(category: str, amount: str) -> str:
    """Adjust a category amount in the current plan."""
    plan = budget_context.get("plan")
    if plan is None:
        return "No plan yet. Ask to 'propose budget' first."
    if category not in CATEGORIES:
        return f"Unknown category. Choose from: {', '.join(CATEGORIES)}."
    plan["allocations"][category] = max(_parse_float(amount), 0.0)
    budget_context["plan"] = plan
    _save_memory(budget_context)
    return render_plan(plan)

@tool
def clear_goals_tool() -> str:
    """Remove all currently saved goals."""
    budget_context["goals"] = []
    _save_memory(budget_context)
    return "Cleared all goals."

@tool
def show_plan_tool() -> str:
    """Show the current budget plan if available."""
    plan = budget_context.get("plan")
    if not plan:
        return "No plan yet. Ask to 'propose budget' first."
    return render_plan(plan)

@tool
def list_goals_tool() -> str:
    """List all saved goals from memory."""
    goals = budget_context.get("goals", [])
    if not goals:
        return "No goals saved yet."
    lines = ["Saved goals:"]
    for g in goals:
        lines.append(f"- {g.name}: target {g.target_amount:.2f} in {g.months} months (priority {g.priority})")
    return "\n".join(lines)

@tool
def show_profile_tool() -> str:
    """Show the current saved profile."""
    p = budget_context.get("profile")
    if not p:
        return "No profile saved yet."
    return f"Profile: net_income={p.net_income}, cost_of_living={p.cost_of_living}, household_size={p.household_size}"

budget_tools = [
    set_profile_tool,
    add_goal_tool,
    propose_budget_tool,
    adjust_category_tool,
    clear_goals_tool,
    show_plan_tool,
    list_goals_tool,
    show_profile_tool,
]

# Prebuild a tool runner instance we can call from a node
_tools_runner = ToolNode(budget_tools)

# Message-based state for ToolNode flow
class MsgState(TypedDict):
    messages: List
    tool_calls_made: int


def _format_memory_for_prompt() -> str:
    p = budget_context.get("profile")
    goals = budget_context.get("goals", [])
    plan = budget_context.get("plan")
    parts = ["You have access to persistent episodic memory. Current saved state:"]
    if p:
        parts.append(f"- Profile: net_income={p.net_income}, cost_of_living={p.cost_of_living}, household_size={p.household_size}")
    if goals:
        parts.append("- Goals:")
        for g in goals:
            parts.append(f"  • {g.name}: target {g.target_amount} in {g.months} months (priority {g.priority})")
    if plan:
        parts.append("- A budget plan exists for the current profile/goals. Use show_plan_tool to display it.")
    if not p and not goals and not plan:
        parts.append("- No saved profile, goals, or plan.")
    parts.append("Use this memory as context when reasoning and responding.")
    return "\n".join(parts)

# Model node that can request tools (disabled after first tool call)
def budget_model_node(state: MsgState) -> MsgState:
    tools_allowed = state.get("tool_calls_made", 0) < 1
    model = LLM if not tools_allowed else LLM.bind_tools(budget_tools)
    # Inject memory summary for model context without mutating the persisted chat history
    messages_to_model = list(state["messages"]) or []
    try:
        mem_msg = SystemMessage(content=_format_memory_for_prompt())
        # Insert after the first SystemMessage if present, else prepend
        inserted = False
        for i, m in enumerate(messages_to_model):
            if isinstance(m, SystemMessage):
                messages_to_model.insert(i + 1, mem_msg)
                inserted = True
                break
        if not inserted:
            messages_to_model = [mem_msg] + messages_to_model
    except Exception:
        pass
    ai = model.invoke(messages_to_model)  # returns AIMessage with optional tool_calls
    return {"messages": state["messages"] + [ai], "tool_calls_made": state.get("tool_calls_made", 0)}

# Tools node that executes tools and increments the counter
def tools_node(state: MsgState) -> MsgState:
    result = _tools_runner.invoke(state)
    return {
        "messages": result["messages"],
        "tool_calls_made": state.get("tool_calls_made", 0) + 1,
    }

# Routing: if the last AI message has tool calls, go to tools; else end
def route_tools(state: MsgState):
    if state.get("tool_calls_made", 0) >= 1:
        return END
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and getattr(last, "tool_calls", None):
        return "tools"
    return END

# Build tool-enabled graph
from langgraph.graph import START

tool_graph = StateGraph(MsgState)
tool_graph.add_node("model", budget_model_node)
tool_graph.add_node("tools", tools_node)

tool_graph.add_edge(START, "model")
tool_graph.add_conditional_edges("model", route_tools)
tool_graph.add_edge("tools", "model")  # loop back for a single natural-language reply

tool_app = tool_graph.compile()

SYSTEM_PROMPT = (
  """
    You are a budgeting assistant with persistent episodic memory saved on disk.
    - Only call a tool if strictly necessary to read/change state (profile, goals, plan) or to compute/propose/adjust.
    - After a successful tool call, reply to the user in natural language and DO NOT call any more tools in this turn.
    - Prefer using list_goals_tool and show_profile_tool when the user asks to view saved state.
  """
)

def call_budget_agent_tools(user_input: str, recursion_limit: int = 25):
    state = {"messages": [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_input)
    ], "tool_calls_made": 0}
    final = tool_app.invoke(state, config={"recursion_limit": recursion_limit, "debug": True})
    ai_msgs = [m for m in final["messages"] if isinstance(m, AIMessage)]
    text = ai_msgs[-1].content if ai_msgs else "Done."
    print(text)
    return text, final

In [ ]:
call_budget_agent_tools("create a profile for a single professional in high cost of living city") 

In [ ]:
# Interactive chat for the LangGraph + tools budget agent
from IPython.display import display, clear_output
import ipywidgets as widgets
from langchain_core.messages import HumanMessage, AIMessage


def interactive_budget_chat():
    state = {"messages": [], "tool_calls_made": 0}

    input_box = widgets.Text(
        description='You:',
        placeholder='Type your message (e.g., my income is 120000, add goal travel 200000 in 6 months)',
        layout=widgets.Layout(width='90%')
    )
    send_btn = widgets.Button(description='Send', button_style='primary')
    reset_btn = widgets.Button(description='Reset Chat', button_style='warning')
    output_area = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='8px'))

    def render_conversation():
        with output_area:
            clear_output(wait=True)
            for m in state["messages"]:
                if isinstance(m, HumanMessage):
                    print(f"You: {m.content}")
                elif isinstance(m, AIMessage):
                    print(f"Assistant: {m.content}")

    def do_send(_=None):
        text = input_box.value.strip()
        if not text:
            return
        # Add user message
        state["messages"].append(HumanMessage(content=text))
        # Run the tool-enabled app (single tool call per turn enforced by graph)
        final = tool_app.invoke(state)
        # Update state with full message history returned by the graph
        state["messages"] = final.get("messages", state["messages"])
        # Reset tool_calls_made so next user turn can call tools again
        state["tool_calls_made"] = 0
        render_conversation()
        input_box.value = ''

    def do_reset(_):
        state["messages"] = []
        state["tool_calls_made"] = 0
        # Optionally also clear in-memory budget context
        budget_context["goals"] = []
        budget_context["plan"] = None
        budget_context["profile"] = None
        render_conversation()

    input_box.on_submit(do_send)
    send_btn.on_click(do_send)
    reset_btn.on_click(do_reset)

    controls = widgets.HBox([send_btn, reset_btn])
    ui = widgets.VBox([output_area, input_box, controls])
    display(ui)
    print("Tip: you can say things like 'my income is 120000, high COL, household 1', 'add goal emergency fund 600000 in 12 months priority 1', 'propose budget', 'set Groceries to 15000', 'show plan'.")

# To launch the chat:
# interactive_budget_chat()


In [ ]:
interactive_budget_chat()

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode


def _normalize_category(name: str) -> str | None:
    s = (name or "").strip().lower()
    if not s:
        return None
    synonyms = {
        "shopping": "Shopping & Entertainment",
        "entertainment": "Shopping & Entertainment",
        "shopping & entertainment": "Shopping & Entertainment",
        "subs": "Subscriptions",
        "subscription": "Subscriptions",
        "subscriptions": "Subscriptions",
        "transportation": "Transport",
        "grocer": "Groceries",
        "grocery": "Groceries",
        "healthcare": "Health",
    }
    if s in synonyms:
        return synonyms[s]
    for cat in CATEGORIES:
        if cat.lower() == s:
            return cat
    for cat in CATEGORIES:
        if s in cat.lower() or cat.lower() in s:
            return cat
    return None


@tool
def rebalance_category_tool(category: str, amount: str) -> str:
    """Set a category to a new amount and proportionally adjust other categories to keep total constant, respecting floors."""
    plan = budget_context.get("plan")
    if plan is None:
        return "No plan yet. Ask to 'propose budget' first."

    allocations = plan.get("allocations", {})
    income = float(plan.get("income", 0.0))

    cat = _normalize_category(category)
    if not cat or cat not in allocations:
        return f"Unknown category. Choose from: {', '.join(CATEGORIES)}."

    new_amt = max(_parse_float(amount), 0.0)
    old_amt = float(allocations.get(cat, 0.0))
    delta = new_amt - old_amt

    if abs(delta) < 1e-6:
        return render_plan(plan)

    # Apply the new target first
    allocations[cat] = new_amt

    note = ""
    if delta > 0:
        # Need to reduce other categories to offset the increase
        remaining = delta
        prioritized = [c for c in ELASTICITY_ORDER if c in allocations and c != cat]
        others = [c for c in allocations.keys() if c != cat and c not in prioritized]
        order = prioritized + others
        for c in order:
            if remaining <= 1e-6:
                break
            floor_pct = float(FLOORS.get(c, 0.0))
            floor_amt = income * (floor_pct / 100.0)
            available = max(allocations[c] - floor_amt, 0.0)
            take = min(remaining, available)
            allocations[c] = allocations[c] - take
            remaining -= take
        if remaining > 1e-6:
            note = f"Note: hit category floors; unable to fully offset increase by {remaining:.2f}."
    else:
        # Decrease target; distribute freed funds proportionally to other categories
        freed = -delta
        other_cats = [c for c in allocations.keys() if c != cat]
        total_other = sum(allocations[c] for c in other_cats)
        if total_other > 1e-6:
            for c in other_cats:
                share = allocations[c] / total_other
                allocations[c] = allocations[c] + freed * share

    budget_context["plan"] = plan
    _save_memory(budget_context)
    out = render_plan(plan)
    if note:
        out = out + "\n" + note
    return out

# Register new tool and refresh tool runner so it is available immediately
if rebalance_category_tool not in budget_tools:
    budget_tools.append(rebalance_category_tool)
_tools_runner = ToolNode(budget_tools)

